# ERA5 Daily TX Percentile Thresholds — ETCCDI TX90p Method

Computes calendar-day temperature percentile thresholds from ERA5 daily maximum
temperature (TX) following the ETCCDI TX90p methodology (Zhang et al., 2005).

**Key steps:**
1. Load ERA5 daily TX over the full time series
2. Compute 90th / 95th / 99th calendar-day thresholds from the 1981–2010 baseline
   using a 5-day centred moving window and the Zhang et al. (2005) bootstrap for
   in-base-period years
3. Export thresholds to NetCDF
4. Interactive explorer: choose a city/ROI by coordinates and a percentile to display
   the raw TX series alongside the selected threshold

**Reference:** Zhang, X. et al. (2005). Avoiding inhomogeneity in percentile-based
indices of temperature extremes. *J. Climate*, 18, 1641–1651.
doi:[10.1175/JCLI3366.1](https://doi.org/10.1175/JCLI3366.1)

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — all editable parameters live here
# ══════════════════════════════════════════════════════════════════════════════

# ── Data paths ────────────────────────────────────────────────────────────────
ERA5_TX_PATH      = "../data/era5_tx_daily.nc"   # ERA5 daily TX NetCDF (downloaded below)
TX_VARNAME        = "mx2t"                         # variable name written to / read from file
KELVIN_INPUT      = True                           # True — ERA5 TX is in Kelvin
OUTPUT_DIR        = "../outputs"

# ── Default city / ROI ────────────────────────────────────────────────────────
DEFAULT_CITY      = "Salvador, Brazil"
DEFAULT_LAT       = -12.97            # decimal degrees N
DEFAULT_LON       = -38.51            # decimal degrees E

# ── GEE data acquisition ──────────────────────────────────────────────────────
GEE_PROJECT       = "tl-cities"       # same project used in other notebooks
ROI_BUFFER_DEG    = 1.0               # degrees of padding around DEFAULT_LAT / DEFAULT_LON
GEE_ERA5_START    = "1979-01-01"      # inclusive — ERA5 daily starts 1979-01-02 on GEE
GEE_ERA5_END      = "2024-01-01"      # exclusive end date

# ── Baseline period (ETCCDI standard) ────────────────────────────────────────
BASELINE_START    = 1981
BASELINE_END      = 2010

# ── Moving-window half-width (days) ──────────────────────────────────────────
# WINDOW_HALF = 2  →  5-day centred window (d−2 … d … d+2)
WINDOW_HALF       = 2

# ── Percentiles to compute ────────────────────────────────────────────────────
PERCENTILES       = [90, 95, 99]

# ── NetCDF export ─────────────────────────────────────────────────────────────
THRESHOLD_NC_PATH = (
    f"{OUTPUT_DIR}/era5_tx_thresholds_{BASELINE_START}_{BASELINE_END}.nc"
)

In [24]:
# ── City / ROI selector ───────────────────────────────────────────────────────
# Change SELECTED_CITY to switch the analysis region, then re-run all cells.

CITIES = {
    "Salvador, Brazil": {"lat": -12.97, "lon": -38.51},
    "Bengaluru, India": {"lat":  12.97, "lon":  77.59},
}

SELECTED_CITY = "Bengaluru, India"   # ← change this to switch city

DEFAULT_CITY = SELECTED_CITY
DEFAULT_LAT  = CITIES[SELECTED_CITY]["lat"]
DEFAULT_LON  = CITIES[SELECTED_CITY]["lon"]

# Derive city-specific file paths
_city_slug        = DEFAULT_CITY.lower().replace(", ", "_").replace(" ", "_")
ERA5_TX_PATH      = f"../data/era5_tx_daily_{_city_slug}.nc"
THRESHOLD_NC_PATH = f"{OUTPUT_DIR}/era5_tx_thresholds_{_city_slug}_{BASELINE_START}_{BASELINE_END}.nc"

print(f"Selected city  : {DEFAULT_CITY}  (lat={DEFAULT_LAT}, lon={DEFAULT_LON})")
print(f"ERA5 data path : {ERA5_TX_PATH}")

Selected city  : Bengaluru, India  (lat=12.97, lon=77.59)
ERA5 data path : ../data/era5_tx_daily_bengaluru_india.nc


In [11]:
import warnings
from pathlib import Path
from datetime import date as _date

import ee
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"xarray  {xr.__version__}")
print(f"numpy   {np.__version__}")
print(f"Output  {Path(OUTPUT_DIR).resolve()}")

xarray  2024.7.0
numpy   1.26.4
Output  /Users/martynclark/heatInsights-notebooks/outputs


## 1. Acquire ERA5 Daily TX from Google Earth Engine

Downloads `maximum_2m_temperature` from [`ECMWF/ERA5/DAILY`](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_DAILY)
for the bounding box centred on `DEFAULT_LAT` / `DEFAULT_LON` and saves it to
`ERA5_TX_PATH` as a compressed NetCDF.

Uses the same GEE project (`tl-cities`) and credentials as the other notebooks
in this workspace — no extra setup required.

- Values are in **Kelvin** as stored in GEE; `KELVIN_INPUT = True` in the config
  cell handles the K → °C conversion in the load step below.
- Data are fetched **year-by-year** to stay within GEE's `getRegion()` response
  limits (~100 MB per call).
- The cell is **idempotent**: if `ERA5_TX_PATH` already exists the download is
  skipped entirely.

In [26]:
_ERA5_DAILY_ID   = "ECMWF/ERA5/DAILY"
_ERA5_TX_BAND    = "maximum_2m_air_temperature"   # Kelvin
_ERA5_SCALE      = 27830                           # native resolution in metres (~0.25 deg)
_PROVENANCE_PATH = ERA5_TX_PATH.replace(".nc", "_acquisition.txt")

# ── Print provenance from last acquisition (if it exists) ────────────────────
if Path(_PROVENANCE_PATH).exists():
    print("Last acquisition parameters (compare with current config):")
    print("-" * 60)
    print(Path(_PROVENANCE_PATH).read_text())
    print("-" * 60)

def _era5_file_valid(path, varname):
    """Return True only if the file exists and contains varname."""
    if not Path(path).exists():
        return False
    try:
        ds = xr.open_dataset(path)
        ok = varname in ds
        ds.close()
        return ok
    except Exception:
        return False

if _era5_file_valid(ERA5_TX_PATH, TX_VARNAME):
    print(f"ERA5 file already exists and is valid at {ERA5_TX_PATH} — skipping download.")
    print("Delete the .nc (and optionally the .txt) to force a fresh download.")

else:
    if Path(ERA5_TX_PATH).exists():
        print(f"Found incomplete file at {ERA5_TX_PATH} (missing '{TX_VARNAME}') — deleting and re-downloading.")
        Path(ERA5_TX_PATH).unlink()
        if Path(_PROVENANCE_PATH).exists():
            Path(_PROVENANCE_PATH).unlink()
    # Initialise GEE
    try:
        ee.Initialize(project=GEE_PROJECT)
        print(f"GEE initialised  (project: {GEE_PROJECT})")
    except Exception as e:
        raise RuntimeError(
            "GEE initialisation failed: " + str(e) + "\n"
            "Run ee.Authenticate() once if you have not already done so."
        ) from e

    _lon_min = DEFAULT_LON - ROI_BUFFER_DEG
    _lat_min = DEFAULT_LAT - ROI_BUFFER_DEG
    _lon_max = DEFAULT_LON + ROI_BUFFER_DEG
    _lat_max = DEFAULT_LAT + ROI_BUFFER_DEG

    roi = ee.Geometry.Rectangle([_lon_min, _lat_min, _lon_max, _lat_max])
    print(f"ROI  : lon [{_lon_min:.2f}, {_lon_max:.2f}]  lat [{_lat_min:.2f}, {_lat_max:.2f}]")

    # Base collection — .select() applied per-year inside the loop.
    # Applying it before filterDate causes GEE to raise "No bands in collection"
    # when a per-year sub-collection is empty.
    base_col = (
        ee.ImageCollection(_ERA5_DAILY_ID)
          .filterDate(GEE_ERA5_START, GEE_ERA5_END)
          .filterBounds(roi)
    )
    print(f"Images in collection : {base_col.size().getInfo()}")

    start_yr = int(GEE_ERA5_START[:4])
    end_yr   = int(GEE_ERA5_END[:4])

    frames  = []
    skipped = []

    for yr in range(start_yr, end_yr):
        try:
            region_rows = (
                base_col
                  .filterDate(f"{yr}-01-01", f"{yr + 1}-01-01")
                  .select([_ERA5_TX_BAND])
                  .getRegion(roi, _ERA5_SCALE)
                  .getInfo()
            )
        except Exception as e:
            print(f"  {yr}: skipped — {e}")
            skipped.append(yr)
            continue

        header  = region_rows[0]
        records = region_rows[1:]
        if not records:
            skipped.append(yr)
            continue

        df_yr = pd.DataFrame(records, columns=header)
        df_yr = df_yr.dropna(subset=[_ERA5_TX_BAND])
        df_yr["time"] = pd.to_datetime(df_yr["time"], unit="ms").dt.normalize()
        frames.append(df_yr)
        print(f"  {yr}: {len(df_yr):>6,} pixel-day rows", end="\r")

    print(
        f"\nDownload complete — {sum(len(f) for f in frames):,} total rows"
        + (f"  ({len(skipped)} years skipped: {skipped})" if skipped else "")
    )

    df = (
        pd.concat(frames, ignore_index=True)
          .rename(columns={_ERA5_TX_BAND: "tx", "longitude": "lon", "latitude": "lat"})
          [["time", "lat", "lon", "tx"]]
          .drop_duplicates(subset=["time", "lat", "lon"])
    )

    lats  = np.sort(df["lat"].unique())
    lons  = np.sort(df["lon"].unique())
    times = np.sort(df["time"].unique())

    full_idx = pd.MultiIndex.from_product(
        [times, lats, lons], names=["time", "lat", "lon"]
    )
    arr = (
        df.set_index(["time", "lat", "lon"])["tx"]
          .reindex(full_idx)
          .values
          .reshape(len(times), len(lats), len(lons))
          .astype(np.float32)
    )

    tx_gee = xr.DataArray(
        arr,
        dims=["time", "lat", "lon"],
        coords={"time": pd.DatetimeIndex(times), "lat": lats, "lon": lons},
        attrs={
            "units":     "K",
            "long_name": "Daily maximum 2 m air temperature",
            "source":    f"GEE {_ERA5_DAILY_ID} / {_ERA5_TX_BAND}",
        },
    )

    Path(ERA5_TX_PATH).parent.mkdir(parents=True, exist_ok=True)
    _ds_out = tx_gee.to_dataset(name=TX_VARNAME)
    for _engine, _enc in [
        ("netcdf4",  {TX_VARNAME: {"dtype": "float32", "zlib": True, "complevel": 4}}),
        ("h5netcdf", {TX_VARNAME: {"dtype": "float32", "zlib": True, "complevel": 4}}),
        ("scipy",    {}),
    ]:
        try:
            _ds_out.to_netcdf(ERA5_TX_PATH, engine=_engine, encoding=_enc)
            sz_mb = Path(ERA5_TX_PATH).stat().st_size / 1e6
            print(f"Saved  to  {ERA5_TX_PATH}  ({sz_mb:.1f} MB)  [engine: {_engine}]")
            break
        except (ModuleNotFoundError, ValueError):
            continue
    else:
        raise RuntimeError("No suitable NetCDF engine found. Install netCDF4: pip install netCDF4")
    print(f"Grid   :  {len(lats)} lats x {len(lons)} lons x {len(times)} days")

    # Write provenance record alongside the NetCDF
    provenance_lines = [
        f"Downloaded       : {_date.today()}",
        f"ERA5_TX_PATH     : {ERA5_TX_PATH}",
        f"GEE_PROJECT      : {GEE_PROJECT}",
        f"GEE collection   : {_ERA5_DAILY_ID}",
        f"GEE band         : {_ERA5_TX_BAND}",
        f"GEE_ERA5_START   : {GEE_ERA5_START}",
        f"GEE_ERA5_END     : {GEE_ERA5_END}",
        f"DEFAULT_LAT      : {DEFAULT_LAT}",
        f"DEFAULT_LON      : {DEFAULT_LON}",
        f"ROI_BUFFER_DEG   : {ROI_BUFFER_DEG}",
        f"ROI bbox         : lon [{_lon_min:.4f}, {_lon_max:.4f}]  lat [{_lat_min:.4f}, {_lat_max:.4f}]",
        f"Grid             : {len(lats)} lats x {len(lons)} lons x {len(times)} days",
    ]
    Path(_PROVENANCE_PATH).write_text("\n".join(provenance_lines) + "\n")
    print(f"Provenance written to {_PROVENANCE_PATH}")

GEE initialised  (project: tl-cities)
ROI  : lon [76.59, 78.59]  lat [11.97, 13.97]
Images in collection : 15165
  2021: skipped — ImageCollection.getRegion: No bands in collection.
  2022: skipped — ImageCollection.getRegion: No bands in collection.
  2023: skipped — ImageCollection.getRegion: No bands in collection.

Download complete — 970,560 total rows  (3 years skipped: [2021, 2022, 2023])
Saved  to  ../data/era5_tx_daily_bengaluru_india.nc  (2.3 MB)  [engine: netcdf4]
Grid   :  8 lats x 8 lons x 15165 days
Provenance written to ../data/era5_tx_daily_bengaluru_india_acquisition.txt


## 1. Load ERA5 Daily TX

If the ERA5 file is not found at `ERA5_TX_PATH`, synthetic ERA5-like data for
Salvador, Brazil is generated so that the rest of the notebook runs end-to-end.
Replace `ERA5_TX_PATH` with a real ERA5 download to run the full analysis.

In [27]:
def _normalise_coords(da):
    """Rename 'latitude'/'longitude' -> 'lat'/'lon' if present."""
    renames = {}
    if "latitude"  in da.dims: renames["latitude"]  = "lat"
    if "longitude" in da.dims: renames["longitude"] = "lon"
    return da.rename(renames) if renames else da


SYNTHETIC = False

try:
    ds_raw = xr.open_dataset(ERA5_TX_PATH)

    if TX_VARNAME not in ds_raw:
        ds_raw.close()
        raise KeyError(
            f"Variable '{TX_VARNAME}' not found in {ERA5_TX_PATH}. "
            f"Variables present: {list(ds_raw.data_vars)}. "
            "The file is likely incomplete from a failed download — "
            "delete it and re-run the acquisition cell."
        )

    tx = _normalise_coords(ds_raw[TX_VARNAME])
    if KELVIN_INPUT:
        tx = tx - 273.15
    tx.attrs["units"] = "degrees_Celsius"
    tx.attrs["long_name"] = "Daily maximum 2 m temperature"
    print(f"Loaded ERA5 TX from {ERA5_TX_PATH}")

except (FileNotFoundError, KeyError) as err:
    print(f"Could not load ERA5 file: {err}")
    print("Falling back to synthetic data for demonstration.\n")
    SYNTHETIC = True

    rng      = np.random.default_rng(42)
    time_idx = pd.date_range("1979-01-01", "2023-12-31", freq="D")
    lats     = np.linspace(-13.25, -12.75, 5)
    lons     = np.linspace(-38.75, -38.25, 5)

    doy_arr  = time_idx.day_of_year.values[:, None, None].astype(np.float32)
    yr_arr   = time_idx.year.values[:, None, None].astype(np.float32)

    seasonal = 28.0 + 4.5 * np.cos(2 * np.pi * (doy_arr - 20) / 365)
    trend    = 0.025 * (yr_arr - 1979)
    lat_grad = -0.4 * (lats[None, :, None] - lats.mean())
    noise    = rng.normal(0, 1.8, (len(time_idx), len(lats), len(lons))).astype(np.float32)

    tx = xr.DataArray(
        (seasonal + trend + lat_grad + noise).astype(np.float32),
        dims=["time", "lat", "lon"],
        coords={"time": time_idx, "lat": lats, "lon": lons},
        attrs={
            "units":     "degrees_Celsius",
            "long_name": "Daily maximum 2 m temperature (synthetic)",
            "note":      "Replace ERA5_TX_PATH with real ERA5 data for production use.",
        },
    )

print(f"  Time  : {str(tx.time.values[0])[:10]}  ->  {str(tx.time.values[-1])[:10]}")
print(f"  Grid  : {dict(zip(tx.dims, tx.shape))}")
print(f"  TX    : {float(tx.min()):.1f} - {float(tx.max()):.1f} {'K' if not SYNTHETIC and KELVIN_INPUT else 'C'}")
if SYNTHETIC:
    print("  [Synthetic data - results are illustrative only]")

Loaded ERA5 TX from ../data/era5_tx_daily_bengaluru_india.nc
  Time  : 1979-01-02  ->  2020-07-09
  Grid  : {'time': 15165, 'lat': 8, 'lon': 8}
  TX    : 17.9 - 41.0 K


## 2. ETCCDI Threshold Computation

### Method

For each calendar day-of-year $d$ (1 – 365), the threshold at percentile $p$ is:

$$T_p(d) = \mathrm{percentile}_p\!\left(\bigl\{\mathrm{TX}(t)\bigr\}_{\substack{t \in \text{baseline} \\ |\mathrm{DOY}(t)-d| \le 2}}\right)$$

where the pool contains **5 days × 30 years = 150 values** (fewer near the year boundary
due to wrapping).

### Bootstrap (Zhang et al., 2005)

To avoid inflated TX90p counts within the baseline period, a leave-one-out bootstrap
is also computed: for each year $y$ in the baseline the threshold is re-estimated
from the remaining 29 years (145 values).  This bootstrap threshold is stored in
`thresh_boot` (dimensions `year × doy × lat × lon`) and is used when counting
exceedance days inside the baseline period.

All spatial computation is **fully vectorised** — the outer loop runs over
365 DOYs only, never over individual grid cells.

In [28]:
def compute_doy_thresholds(
    tx: xr.DataArray,
    baseline_start: int,
    baseline_end: int,
    window_half: int = 2,
    percentiles: tuple = (90, 95, 99),
    bootstrap: bool = True,
) -> tuple:
    """
    Compute calendar-day TX percentile thresholds following ETCCDI TX90p.

    Implements the Zhang et al. (2005) bootstrap for the in-base period.
    Loops over 365 DOYs only — all spatial computation is vectorised.

    Parameters
    ----------
    tx             : xr.DataArray (time, lat, lon), degrees Celsius
    baseline_start : int   first year of the baseline period
    baseline_end   : int   last  year of the baseline period
    window_half    : int   half-width of the centred window in days (default 2 → 5 days)
    percentiles    : sequence[int]   percentile levels to compute
    bootstrap      : bool  if True, also compute leave-one-out thresholds per
                           baseline year (needed for unbiased TX90p counting)

    Returns
    -------
    thresh_std  : xr.Dataset  dims (doy, lat, lon)         standard thresholds
    thresh_boot : xr.Dataset  dims (year, doy, lat, lon)   bootstrap thresholds,
                              or None if bootstrap=False
    """
    # ── Subset to baseline ────────────────────────────────────────────────────
    bl      = tx.sel(time=slice(str(baseline_start), str(baseline_end)))
    bl_doy  = bl.time.dt.dayofyear.values.astype(np.int16)   # (T,)
    bl_year = bl.time.dt.year.values.astype(np.int16)         # (T,)
    bl_vals = bl.values.astype(np.float32)                    # (T, nlat, nlon)

    n_lat, n_lon   = bl_vals.shape[1], bl_vals.shape[2]
    lat_coord      = bl.lat.values
    lon_coord      = bl.lon.values
    base_years     = np.arange(baseline_start, baseline_end + 1, dtype=np.int16)

    # Pre-allocate output arrays
    std_arrs  = {p: np.full((365, n_lat, n_lon), np.nan, np.float32)
                 for p in percentiles}
    boot_arrs = ({p: np.full((len(base_years), 365, n_lat, n_lon), np.nan, np.float32)
                  for p in percentiles}
                 if bootstrap else None)

    # Helper: set of DOYs in the window around `doy` (1–365 wrap)
    def _window(doy: int) -> np.ndarray:
        raw = np.arange(doy - window_half, doy + window_half + 1)
        return np.where(raw < 1, raw + 365, np.where(raw > 365, raw - 365, raw))

    # ── Main loop: 365 DOYs, no grid-cell loops ───────────────────────────────
    for doy in range(1, 366):
        win_doys = _window(doy)
        # Mask all baseline time steps whose DOY falls in the window.
        # Leap-day (DOY 366) is excluded to keep a strict 365-day calendar.
        win_mask = np.isin(bl_doy, win_doys) & (bl_doy != 366)

        sample_all = bl_vals[win_mask]          # (n_samples, nlat, nlon)

        for p in percentiles:
            std_arrs[p][doy - 1] = np.nanpercentile(sample_all, p, axis=0)

        if bootstrap:
            yr_in_win = bl_year[win_mask]       # year label for each sample row
            for yi, yr in enumerate(base_years):
                # Leave year `yr` out of the pool
                loo = sample_all[yr_in_win != yr]
                for p in percentiles:
                    boot_arrs[p][yi, doy - 1] = np.nanpercentile(loo, p, axis=0)

        if doy % 73 == 0 or doy == 365:
            print(f"  DOY {doy:3d}/365", end="\r")

    print("  DOY 365/365 — building datasets …")

    # ── Package as xr.Datasets ────────────────────────────────────────────────
    doy_coord   = np.arange(1, 366)
    method_base = (
        f"{2 * window_half + 1}-day centred window  |  "
        f"{baseline_start}–{baseline_end} baseline  |  "
        f"Zhang et al. (2005) bootstrap"
    )

    def _build_ds(arrays, extra_dims=None, extra_coords=None):
        data_vars = {}
        for p, arr in arrays.items():
            dims   = (extra_dims or []) + ["doy", "lat", "lon"]
            coords = {"doy": doy_coord, "lat": lat_coord, "lon": lon_coord}
            if extra_coords:
                coords.update(extra_coords)
            data_vars[f"tx{p}p"] = xr.DataArray(
                arr, dims=dims, coords=coords,
                attrs={
                    "long_name": (
                        f"ERA5 daily TX {p}th-percentile threshold "
                        f"(calendar day, {baseline_start}–{baseline_end})"
                    ),
                    "units":     "degrees_Celsius",
                    "method":    f"ETCCDI TX{p}p  |  " + method_base,
                    "reference": "Zhang et al. (2005), J. Climate, doi:10.1175/JCLI3366.1",
                },
            )
        return xr.Dataset(
            data_vars,
            attrs={
                "title":           f"ERA5 TX percentile thresholds ({baseline_start}–{baseline_end})",
                "baseline_period": f"{baseline_start}–{baseline_end}",
                "window":          f"{2*window_half+1}-day centred (±{window_half} days)",
                "percentiles":     str(list(arrays.keys())),
                "conventions":     "CF-1.8",
            },
        )

    thresh_std  = _build_ds(std_arrs)
    thresh_boot = (
        _build_ds(
            boot_arrs,
            extra_dims=["year"],
            extra_coords={"year": base_years.astype(int)},
        )
        if bootstrap else None
    )

    return thresh_std, thresh_boot

## 3. Compute Thresholds

In [29]:
print("Computing ETCCDI TX percentile thresholds …")
print(f"  Baseline : {BASELINE_START}–{BASELINE_END}")
print(f"  Window   : ±{WINDOW_HALF} days  ({2*WINDOW_HALF+1}-day centred)")
print(f"  Levels   : {PERCENTILES}")
print()

thresh_std, thresh_boot = compute_doy_thresholds(
    tx,
    baseline_start = BASELINE_START,
    baseline_end   = BASELINE_END,
    window_half    = WINDOW_HALF,
    percentiles    = tuple(PERCENTILES),
    bootstrap      = True,
)

print()
print(thresh_std)
print()

# ── Quick sanity check at the default location ────────────────────────────
loc = dict(lat=DEFAULT_LAT, lon=DEFAULT_LON, method="nearest")
print(f"Threshold summary at nearest grid cell to {DEFAULT_CITY}:")
for p in PERCENTILES:
    v = thresh_std[f"tx{p}p"].sel(**loc)
    print(
        f"  TX{p}p  annual mean: {float(v.mean()):.1f}°C  "
        f"range: {float(v.min()):.1f}–{float(v.max()):.1f}°C"
    )

Computing ETCCDI TX percentile thresholds …
  Baseline : 1981–2010
  Window   : ±2 days  (5-day centred)
  Levels   : [90, 95, 99]

  DOY 365/365 — building datasets …

<xarray.Dataset> Size: 283kB
Dimensions:  (doy: 365, lat: 8, lon: 8)
Coordinates:
  * doy      (doy) int64 3kB 1 2 3 4 5 6 7 8 ... 358 359 360 361 362 363 364 365
  * lat      (lat) float64 64B 12.13 12.38 12.63 12.88 13.13 13.38 13.63 13.88
  * lon      (lon) float64 64B 76.63 76.88 77.13 77.38 77.63 77.88 78.13 78.38
Data variables:
    tx90p    (doy, lat, lon) float32 93kB 28.44 29.29 30.22 ... 28.48 28.67
    tx95p    (doy, lat, lon) float32 93kB 28.72 29.59 30.47 ... 28.79 29.36
    tx99p    (doy, lat, lon) float32 93kB 29.26 30.3 31.76 ... 30.21 29.72 30.12
Attributes:
    title:            ERA5 TX percentile thresholds (1981–2010)
    baseline_period:  1981–2010
    window:           5-day centred (±2 days)
    percentiles:      [90, 95, 99]
    conventions:      CF-1.8

Threshold summary at nearest grid cell to 

## 4. Export Thresholds to NetCDF

In [30]:
encoding = {
    var: {"dtype": "float32", "zlib": True, "complevel": 4}
    for var in thresh_std.data_vars
}

thresh_std.to_netcdf(THRESHOLD_NC_PATH, encoding=encoding)

nc_size = Path(THRESHOLD_NC_PATH).stat().st_size / 1e6
print(f"Standard thresholds  →  {THRESHOLD_NC_PATH}")
print(f"File size : {nc_size:.2f} MB")
print(f"Dimensions: {dict(thresh_std.dims)}")
print()
print("Variables:")
for var in thresh_std.data_vars:
    print(f"  {var}: {thresh_std[var].attrs['long_name']}")

Standard thresholds  →  ../outputs/era5_tx_thresholds_bengaluru_india_1981_2010.nc
File size : 0.20 MB
Dimensions: {'doy': 365, 'lat': 8, 'lon': 8}

Variables:
  tx90p: ERA5 daily TX 90th-percentile threshold (calendar day, 1981–2010)
  tx95p: ERA5 daily TX 95th-percentile threshold (calendar day, 1981–2010)
  tx99p: ERA5 daily TX 99th-percentile threshold (calendar day, 1981–2010)


## 5. Interactive TX vs. Threshold Explorer

Set a location by latitude / longitude (nearest ERA5 grid cell is selected
automatically), choose a percentile level, and set a year range.  The chart
shows the raw daily TX series in blue and the calendar-day threshold as a
dashed red line, with exceedance days shaded.

In [37]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _threshold_on_dates(thresh_doy_1d, time_index):
    """Map a (365,) DOY array onto a real date axis; leap day maps to DOY 365."""
    doys = np.clip(time_index.day_of_year.values, 1, 365)
    return thresh_doy_1d[doys - 1]


def plot_tx_dashboard(lat, lon, percentile, year_start, year_end):
    """
    Two-panel interactive plotly figure:
      Upper — daily TX time series vs calendar-day threshold with hover tooltips
              and exceedance shading for the selected year range.
      Lower — bar chart of annual exceedance-day counts across the full record,
              with the selected window highlighted.
    """
    loc       = dict(lat=lat, lon=lon, method="nearest")
    thresh_key = f"tx{percentile}p"
    thresh_doy = thresh_std[thresh_key].sel(**loc).values   # (365,)

    actual_lat = float(thresh_std["lat"].sel(lat=lat, method="nearest"))
    actual_lon = float(thresh_std["lon"].sel(lon=lon, method="nearest"))

    # ── Upper panel: time-series for the selected window ─────────────────────
    tx_loc    = tx.sel(**loc).sel(time=slice(str(year_start), str(year_end)))
    time_idx  = pd.DatetimeIndex(tx_loc.time.values)
    tx_vals   = tx_loc.values.astype(float)
    thresh_ts = _threshold_on_dates(thresh_doy, time_idx)
    exceed_mask = tx_vals > thresh_ts

    n_exceed = int(exceed_mask.sum())
    n_total  = int((~np.isnan(tx_vals)).sum())
    pct_str  = f"{100 * n_exceed / max(n_total, 1):.1f}%"

    # ── Lower panel: annual counts across the full record ────────────────────
    tx_full      = tx.sel(**loc).to_series().dropna()
    doys_full    = np.clip(tx_full.index.day_of_year.values, 1, 365)
    thresh_full  = thresh_doy[doys_full - 1]
    exceed_full  = pd.Series(tx_full.values > thresh_full, index=tx_full.index)
    annual_counts = exceed_full.groupby(exceed_full.index.year).sum().astype(int)
    expected_days = round(365 * (1 - percentile / 100), 1)

    # ── Build figure ──────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.60, 0.40],
        subplot_titles=(
            f"Daily TX  vs  TX{percentile}p threshold  —  "
            f"({actual_lat:.2f}°N, {actual_lon:.2f}°E)  {year_start}–{year_end}  |  "
            f"Exceedance: {n_exceed} / {n_total} days  ({pct_str})",
            f"Annual days above TX{percentile}p  —  full record",
        ),
        vertical_spacing=0.13,
    )

    # ── Trace 1: raw TX ───────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=time_idx, y=tx_vals,
        mode="lines", name="Daily TX (ERA5)",
        line=dict(color="#4393c3", width=0.9),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>TX: %{y:.1f} °C<extra></extra>",
    ), row=1, col=1)

    # ── Trace 2: percentile threshold ─────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=time_idx, y=thresh_ts,
        mode="lines", name=f"TX{percentile}p threshold",
        line=dict(color="#d6604d", width=2, dash="dash"),
        hovertemplate="<b>%{x|%d %b %Y}</b><br>TX{p}p: %{{y:.1f}} °C<extra></extra>".replace("{p}", str(percentile)),
    ), row=1, col=1)

    # ── Traces 3 & 4: exceedance fill ────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=time_idx, y=thresh_ts,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=time_idx, y=np.where(exceed_mask, tx_vals, thresh_ts),
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(214,96,77,0.30)",
        name="Exceedance", hoverinfo="skip",
    ), row=1, col=1)

    # ── Trace 5: annual bar chart ─────────────────────────────────────────────
    bar_colors = [
        "#d6604d" if year_start <= yr <= year_end else "#aec7e8"
        for yr in annual_counts.index
    ]
    fig.add_trace(go.Bar(
        x=annual_counts.index,
        y=annual_counts.values,
        name=f"Days > TX{percentile}p",
        marker_color=bar_colors,
        hovertemplate=f"<b>%{{x}}</b><br>Days above TX{percentile}p: %{{y}}<extra></extra>",
    ), row=2, col=1)

    # Expected days reference line
    fig.add_hline(
        y=expected_days,
        line_dash="dot", line_color="#888", line_width=1.5,
        annotation_text=f"Expected ~{expected_days:.0f} d/yr ({100 - percentile}%)",
        annotation_font_size=11,
        annotation_position="top right",
        row=2, col=1,
    )

    # ── Layout ────────────────────────────────────────────────────────────────
    fig.update_layout(
        height=780,
        legend=dict(orientation="h", yanchor="bottom", y=1.01,
                    xanchor="right", x=1, font=dict(size=11)),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=65, r=30, t=85, b=45),
        hovermode="x unified",
        bargap=0.25,
    )
    fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
    fig.update_yaxes(showgrid=True, gridcolor="#eeeeee")
    fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
    fig.update_yaxes(title_text="Days / year",       row=2, col=1)

    if SYNTHETIC:
        fig.add_annotation(
            text="Synthetic data — illustrative only",
            xref="paper", yref="paper", x=0.01, y=0.01,
            showarrow=False, font=dict(size=9, color="grey"), opacity=0.7,
        )

    display(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────
_lat_min = float(tx.lat.min())
_lat_max = float(tx.lat.max())
_lon_min = float(tx.lon.min())
_lon_max = float(tx.lon.max())
_yr_min  = int(tx.time.dt.year.min())
_yr_max  = int(tx.time.dt.year.max())

_style = {"description_width": "100px"}

w_city = widgets.Dropdown(
    options=["(manual)"] + list(CITIES.keys()),
    value=DEFAULT_CITY if DEFAULT_CITY in CITIES else "(manual)",
    description="City preset:", style=_style, layout=widgets.Layout(width="340px"),
)
w_lat = widgets.BoundedFloatText(
    value=DEFAULT_LAT, min=_lat_min, max=_lat_max, step=0.01,
    description="Latitude °N:",  style=_style, layout=widgets.Layout(width="240px"),
)
w_lon = widgets.BoundedFloatText(
    value=DEFAULT_LON, min=_lon_min, max=_lon_max, step=0.01,
    description="Longitude °E:", style=_style, layout=widgets.Layout(width="240px"),
)

def _on_city_change(change):
    if change["new"] in CITIES:
        city = CITIES[change["new"]]
        w_lat.value = np.clip(city["lat"], _lat_min, _lat_max)
        w_lon.value = np.clip(city["lon"], _lon_min, _lon_max)

w_city.observe(_on_city_change, names="value")

w_pct = widgets.Dropdown(
    options=[("90th percentile (TX90p)", 90),
             ("95th percentile (TX95p)", 95),
             ("99th percentile (TX99p)", 99)],
    value=90,
    description="Percentile:", style=_style, layout=widgets.Layout(width="340px"),
)
w_yrs = widgets.IntRangeSlider(
    value=[2000, 2020], min=_yr_min, max=_yr_max, step=1,
    description="Year range:", style=_style, layout=widgets.Layout(width="520px"),
    continuous_update=False,
)
w_btn = widgets.Button(
    description="Update plot", button_style="primary",
    layout=widgets.Layout(width="140px"),
)
w_out = widgets.Output()


def _render(_=None):
    with w_out:
        w_out.clear_output(wait=True)
        plot_tx_dashboard(
            lat        = w_lat.value,
            lon        = w_lon.value,
            percentile = w_pct.value,
            year_start = w_yrs.value[0],
            year_end   = w_yrs.value[1],
        )


w_btn.on_click(_render)

ui = widgets.VBox(
    [
        widgets.HTML("<b style='font-size:14px'>ERA5 TX Percentile Threshold Explorer</b>"),
        w_city,
        widgets.HBox([w_lat, w_lon]),
        widgets.HBox([w_pct, w_yrs]),
        w_btn,
        w_out,
    ],
    layout=widgets.Layout(padding="10px"),
)

display(ui)
_render()

## 6. Annual TX Distribution Explorer

Inspect the shape of the daily TX distribution for a selected year at the nearest grid cell.
Vertical dashed lines show the 90th / 95th / 99th percentile thresholds computed above.

## 7. Animated TX Distribution — Year-by-Year

Play through the full record to see how the distribution shifts over time.
The KDE and percentile markers update each frame; the x-axis is fixed so shape changes are directly comparable.

In [35]:
def plot_tx_distribution_animation(lat, lon):
    loc = dict(lat=lat, lon=lon, method="nearest")
    actual_lat = float(tx.lat.sel(lat=lat, method="nearest"))
    actual_lon = float(tx.lon.sel(lon=lon, method="nearest"))

    tx_loc = tx.sel(**loc).to_series().dropna()
    years  = sorted(tx_loc.index.year.unique())

    x_min  = float(tx_loc.min()) - 1
    x_max  = float(tx_loc.max()) + 1
    x_grid = np.linspace(x_min, x_max, 400)

    # Global y ceiling so vertical lines always reach the top of the plot
    all_kde_max = max(
        gaussian_kde(tx_loc[tx_loc.index.year == yr].values, bw_method="scott")(x_grid).max()
        for yr in years
    )
    y_ceil = all_kde_max * 1.15

    def _make_frame_data(yr):
        vals  = tx_loc[tx_loc.index.year == yr].values
        kde_y = gaussian_kde(vals, bw_method="scott")(x_grid)
        counts, edges = np.histogram(vals, bins=40, range=(x_min, x_max), density=True)
        bin_centres   = 0.5 * (edges[:-1] + edges[1:])
        widths        = np.diff(edges)
        emp_pcts      = {p: float(np.percentile(vals, p)) for p in PERCENTILES}
        stats = (
            f"n={len(vals)}  mean={vals.mean():.1f}°C  "
            f"σ={vals.std():.1f}°C  min={vals.min():.1f}°C  max={vals.max():.1f}°C"
        )
        return bin_centres, widths, counts, kde_y, emp_pcts, stats

    def _pct_traces(emp_pcts):
        """Three vertical Scatter traces, one per percentile."""
        traces = []
        for p in PERCENTILES:
            v   = emp_pcts[p]
            col = _THRESH_COLORS[str(p)]
            traces.append(go.Scatter(
                x=[v, v], y=[0, y_ceil],
                mode="lines+text",
                name=f"TX{p}p",
                line=dict(color=col, width=2, dash="dash"),
                text=["", f"TX{p}p {v:.1f}°C"],
                textposition="top right",
                textfont=dict(color=col, size=10),
                hovertemplate=f"TX{p}p: {v:.1f}°C<extra></extra>",
                showlegend=True,
            ))
        return traces

    # ── Build frames ──────────────────────────────────────────────────────────
    frames = []
    for yr in years:
        bx, bw, bc, ky, emp_pcts, stats = _make_frame_data(yr)
        frame_data = [
            go.Bar(x=bx, y=bc, width=bw, name="Daily TX",
                   marker=dict(color="#4393c3", opacity=0.55,
                               line=dict(color="white", width=0.4))),
            go.Scatter(x=x_grid, y=ky, mode="lines", name="KDE",
                       line=dict(color="#2166ac", width=2.5)),
            *_pct_traces(emp_pcts),
        ]
        frames.append(go.Frame(
            data=frame_data,
            name=str(yr),
            layout=go.Layout(title_text=(
                f"Daily TX distribution — {yr}  "
                f"({actual_lat:.2f}°N, {actual_lon:.2f}°E)<br>"
                f"<sup>{stats}</sup>"
            )),
        ))

    # ── Initial frame ─────────────────────────────────────────────────────────
    bx0, bw0, bc0, ky0, emp0, stats0 = _make_frame_data(years[0])

    fig = go.Figure(
        data=[
            go.Bar(x=bx0, y=bc0, width=bw0, name="Daily TX",
                   marker=dict(color="#4393c3", opacity=0.55,
                               line=dict(color="white", width=0.4)),
                   hovertemplate="TX: %{x:.1f}°C<br>Density: %{y:.4f}<extra></extra>"),
            go.Scatter(x=x_grid, y=ky0, mode="lines", name="KDE",
                       line=dict(color="#2166ac", width=2.5),
                       hovertemplate="TX: %{x:.1f}°C<br>Density: %{y:.4f}<extra></extra>"),
            *_pct_traces(emp0),
        ],
        frames=frames,
    )

    sliders = [dict(
        steps=[
            dict(method="animate", args=[[str(yr)],
                 dict(mode="immediate", frame=dict(duration=0, redraw=True),
                      transition=dict(duration=0))],
                 label=str(yr))
            for yr in years
        ],
        active=0,
        currentvalue=dict(prefix="Year: ", font=dict(size=13)),
        pad=dict(t=50), len=0.92, x=0.04,
    )]

    fig.update_layout(
        title=dict(
            text=(
                f"Daily TX distribution — {years[0]}  "
                f"({actual_lat:.2f}°N, {actual_lon:.2f}°E)<br>"
                f"<sup>{stats0}</sup>"
            ),
            font=dict(size=14),
        ),
        xaxis=dict(title="Daily maximum temperature (°C)", range=[x_min, x_max],
                   showgrid=True, gridcolor="#eeeeee"),
        yaxis=dict(title="Probability density", range=[0, y_ceil],
                   showgrid=True, gridcolor="#eeeeee"),
        plot_bgcolor="white", paper_bgcolor="white",
        height=540, bargap=0.05, hovermode="x",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        updatemenus=[dict(
            type="buttons", showactive=False,
            y=0, x=0.04, xanchor="left", yanchor="top", pad=dict(t=10),
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, dict(frame=dict(duration=400, redraw=True),
                                      fromcurrent=True, transition=dict(duration=0))]),
                dict(label="⏸ Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode="immediate", transition=dict(duration=0))]),
            ],
        )],
        sliders=sliders,
        margin=dict(l=65, r=30, t=110, b=100),
    )

    display(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────
_style_a = {"description_width": "100px"}

wa_city = widgets.Dropdown(
    options=["(manual)"] + list(CITIES.keys()),
    value=DEFAULT_CITY if DEFAULT_CITY in CITIES else "(manual)",
    description="City preset:", style=_style_a, layout=widgets.Layout(width="340px"),
)
wa_lat = widgets.BoundedFloatText(
    value=DEFAULT_LAT, min=float(tx.lat.min()), max=float(tx.lat.max()), step=0.01,
    description="Latitude °N:", style=_style_a, layout=widgets.Layout(width="240px"),
)
wa_lon = widgets.BoundedFloatText(
    value=DEFAULT_LON, min=float(tx.lon.min()), max=float(tx.lon.max()), step=0.01,
    description="Longitude °E:", style=_style_a, layout=widgets.Layout(width="240px"),
)

def _on_anim_city_change(change):
    if change["new"] in CITIES:
        city = CITIES[change["new"]]
        wa_lat.value = np.clip(city["lat"], float(tx.lat.min()), float(tx.lat.max()))
        wa_lon.value = np.clip(city["lon"], float(tx.lon.min()), float(tx.lon.max()))

wa_city.observe(_on_anim_city_change, names="value")

wa_btn = widgets.Button(
    description="Build animation", button_style="primary",
    layout=widgets.Layout(width="160px"),
)
wa_out = widgets.Output()

def _render_anim(_=None):
    with wa_out:
        wa_out.clear_output(wait=True)
        print("Building frames …")
        plot_tx_distribution_animation(lat=wa_lat.value, lon=wa_lon.value)

wa_btn.on_click(_render_anim)

ui_anim = widgets.VBox(
    [
        widgets.HTML("<b style='font-size:14px'>TX Distribution Animation</b>"),
        wa_city,
        widgets.HBox([wa_lat, wa_lon]),
        wa_btn,
        wa_out,
    ],
    layout=widgets.Layout(padding="10px"),
)

display(ui_anim)

In [36]:
from scipy.stats import gaussian_kde

_THRESH_COLORS = {"90": "#f4a582", "95": "#d6604d", "99": "#8b1a1a"}


def plot_tx_distribution(lat, lon, year):
    loc = dict(lat=lat, lon=lon, method="nearest")
    actual_lat = float(tx.lat.sel(lat=lat, method="nearest"))
    actual_lon = float(tx.lon.sel(lon=lon, method="nearest"))

    # Pull the year's daily TX values at the selected grid cell
    tx_year = tx.sel(**loc).sel(time=str(year)).values.astype(float)
    tx_year = tx_year[~np.isnan(tx_year)]

    if len(tx_year) == 0:
        print(f"No data for {year}")
        return

    # KDE over the data range
    kde     = gaussian_kde(tx_year, bw_method="scott")
    x_grid  = np.linspace(tx_year.min() - 1, tx_year.max() + 1, 400)
    kde_vals = kde(x_grid)

    # Percentile thresholds: annual mean of the calendar-day threshold for this location
    thresh_vals = {
        p: float(thresh_std[f"tx{p}p"].sel(**loc).mean())
        for p in PERCENTILES
    }

    fig = go.Figure()

    # Histogram (normalised to density)
    fig.add_trace(go.Histogram(
        x=tx_year,
        histnorm="probability density",
        nbinsx=40,
        name="Daily TX",
        marker=dict(color="#4393c3", opacity=0.55, line=dict(color="white", width=0.4)),
        hovertemplate="TX: %{x:.1f} °C<br>Density: %{y:.4f}<extra></extra>",
    ))

    # KDE curve
    fig.add_trace(go.Scatter(
        x=x_grid, y=kde_vals,
        mode="lines", name="KDE",
        line=dict(color="#2166ac", width=2.5),
        hovertemplate="TX: %{x:.1f} °C<br>Density: %{y:.4f}<extra></extra>",
    ))

    # Percentile threshold lines (annual-mean DOY threshold as a reference marker)
    for p in PERCENTILES:
        v = thresh_vals[p]
        col = _THRESH_COLORS[str(p)]
        fig.add_vline(
            x=v,
            line_dash="dash", line_color=col, line_width=2,
            annotation_text=f"TX{p}p ({v:.1f}°C)",
            annotation_font_color=col,
            annotation_font_size=11,
            annotation_position="top right",
        )

    # Summary stats annotation
    stats_text = (
        f"n={len(tx_year)} days  |  "
        f"mean={tx_year.mean():.1f}°C  |  "
        f"σ={tx_year.std():.1f}°C  |  "
        f"min={tx_year.min():.1f}°C  |  "
        f"max={tx_year.max():.1f}°C"
    )

    fig.update_layout(
        title=dict(
            text=(
                f"Daily TX distribution — {year}  "
                f"({actual_lat:.2f}°N, {actual_lon:.2f}°E)<br>"
                f"<sup>{stats_text}</sup>"
            ),
            font=dict(size=14),
        ),
        xaxis_title="Daily maximum temperature (°C)",
        yaxis_title="Probability density",
        plot_bgcolor="white",
        paper_bgcolor="white",
        height=480,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        bargap=0.05,
        hovermode="x",
        margin=dict(l=65, r=30, t=100, b=55),
    )
    fig.update_xaxes(showgrid=True, gridcolor="#eeeeee")
    fig.update_yaxes(showgrid=True, gridcolor="#eeeeee")

    display(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────
_yr_min_dist = int(tx.time.dt.year.min())
_yr_max_dist = int(tx.time.dt.year.max())

_style_d = {"description_width": "100px"}

wd_city = widgets.Dropdown(
    options=["(manual)"] + list(CITIES.keys()),
    value=DEFAULT_CITY if DEFAULT_CITY in CITIES else "(manual)",
    description="City preset:", style=_style_d, layout=widgets.Layout(width="340px"),
)
wd_lat = widgets.BoundedFloatText(
    value=DEFAULT_LAT, min=float(tx.lat.min()), max=float(tx.lat.max()), step=0.01,
    description="Latitude °N:", style=_style_d, layout=widgets.Layout(width="240px"),
)
wd_lon = widgets.BoundedFloatText(
    value=DEFAULT_LON, min=float(tx.lon.min()), max=float(tx.lon.max()), step=0.01,
    description="Longitude °E:", style=_style_d, layout=widgets.Layout(width="240px"),
)

def _on_dist_city_change(change):
    if change["new"] in CITIES:
        city = CITIES[change["new"]]
        wd_lat.value = np.clip(city["lat"], float(tx.lat.min()), float(tx.lat.max()))
        wd_lon.value = np.clip(city["lon"], float(tx.lon.min()), float(tx.lon.max()))

wd_city.observe(_on_dist_city_change, names="value")

wd_year = widgets.IntSlider(
    value=2000, min=_yr_min_dist, max=_yr_max_dist, step=1,
    description="Year:", style=_style_d, layout=widgets.Layout(width="420px"),
    continuous_update=False,
)
wd_btn = widgets.Button(
    description="Update plot", button_style="primary",
    layout=widgets.Layout(width="140px"),
)
wd_out = widgets.Output()


def _render_dist(_=None):
    with wd_out:
        wd_out.clear_output(wait=True)
        plot_tx_distribution(
            lat  = wd_lat.value,
            lon  = wd_lon.value,
            year = wd_year.value,
        )


wd_btn.on_click(_render_dist)

ui_dist = widgets.VBox(
    [
        widgets.HTML("<b style='font-size:14px'>TX Distribution Explorer</b>"),
        wd_city,
        widgets.HBox([wd_lat, wd_lon]),
        wd_year,
        wd_btn,
        wd_out,
    ],
    layout=widgets.Layout(padding="10px"),
)

display(ui_dist)
_render_dist()